## Fengyun-2H satellite

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from xgboost.callback import EarlyStopping
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import timedelta

### Load and preprocess the TLE data


In [2]:
import pandas as pd
# Replace with appropriate path 
df_tles = pd.read_csv('/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/Fengyun-2H.csv', 
                      index_col=0, parse_dates=True)

# Check if datetime index is timezone-naive or timezone-aware
if df_tles.index.tz is None:
    df_tles.index = df_tles.index.tz_localize('UTC')
else:
    df_tles.index = df_tles.index.tz_convert('UTC')

# Now you can safely continue
print(df_tles.describe())
print(df_tles.index.inferred_type)



       eccentricity  argument of perigee  inclination  mean anomaly  \
count   1053.000000          1053.000000  1053.000000   1053.000000   
mean       0.000298             3.384650     0.013876     -3.205598   
std        0.000081             1.664494     0.009192      1.812080   
min        0.000081             0.002272     0.001370     -6.279546   
25%        0.000241             3.390653     0.006065     -4.800076   
50%        0.000297             4.128582     0.011832     -3.241888   
75%        0.000347             4.331120     0.021672     -1.649247   
max        0.000492             6.279838     0.032484     -0.012025   

       Brouwer mean motion  right ascension  
count         1.053000e+03      1053.000000  
mean          4.375018e-03         3.945513  
std           1.098286e-07         1.287210  
min           4.374640e-03         1.758220  
25%           4.374935e-03         2.241507  
50%           4.375026e-03         4.757124  
75%           4.375119e-03         4.8

### Extract and scale Brouwer mean motion

In [3]:
df_element_1 = df_tles[["Brouwer mean motion"]]
df_element_1 = (df_element_1 - df_element_1.mean())*1e7
df_element_1.describe()


,Brouwer mean motion
count,1.053000e+03
mean,7.594568e-12
std,1.098286e+00
min,-3.779637e+00
25%,-8.367223e-01
50%,7.536569e-02
75%,1.009957e+00
max,2.957883e+00


### Visualize the scaled element

In [4]:


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_element_1.index,  
    y=df_element_1[df_element_1.columns[0]],  
    mode="lines", 
    name="Orbital Element"
))

fig.update_layout(
    title="Brouwer mean motion Over Time",
    xaxis_title="Time",
    yaxis_title="Brouwer mean motion",
    template="plotly",
    width=1400
)

fig.show()


### Create lag features for time series forecasting

In [5]:
NUM_LAG_FEATURES = 3

df_y = df_element_1.copy()
df_x = df_element_1.shift(1).rename(columns={"Brouwer mean motion": "bmm_lag_1"})

for lag in range(2, NUM_LAG_FEATURES + 1):
    df_x[f"bmm_lag_{lag}"] = df_element_1.shift(lag)

# Drop rows with NaNs
df_x = df_x.iloc[NUM_LAG_FEATURES:]
df_y = df_y.iloc[NUM_LAG_FEATURES:]



### Split for hyperparameter tuning

In [6]:
# Split for tuning
split_index = int(len(df_x) * 0.8)
split_date = df_x.index[split_index].strftime("%Y-%m-%d")

df_x_train = df_x[:split_date]
df_y_train = df_y[:split_date]
df_x_test = df_x[split_date:]
df_y_test = df_y[split_date:]

# Tune XGBoost model with early stopping
tuned_model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=10,
    random_state=42
)

tuned_model.fit(
    df_x_train,
    df_y_train.values.ravel(),
    eval_set=[(df_x_train, df_y_train), (df_x_test, df_y_test)],
    verbose=True
)

# View best iteration and RMSE
best_n_estimators = tuned_model.best_iteration + 1  
print(f"Best number of trees: {best_n_estimators}")


[0]	validation_0-rmse:0.98258	validation_1-rmse:1.09872
[1]	validation_0-rmse:0.90165	validation_1-rmse:1.01516
[2]	validation_0-rmse:0.82998	validation_1-rmse:0.93917
[3]	validation_0-rmse:0.76650	validation_1-rmse:0.86902
[4]	validation_0-rmse:0.71066	validation_1-rmse:0.81004
[5]	validation_0-rmse:0.66173	validation_1-rmse:0.75710
[6]	validation_0-rmse:0.61889	validation_1-rmse:0.70842
[7]	validation_0-rmse:0.58172	validation_1-rmse:0.66582
[8]	validation_0-rmse:0.54943	validation_1-rmse:0.63108
[9]	validation_0-rmse:0.52138	validation_1-rmse:0.60155
[10]	validation_0-rmse:0.49741	validation_1-rmse:0.57532
[11]	validation_0-rmse:0.47624	validation_1-rmse:0.55246
[12]	validation_0-rmse:0.45800	validation_1-rmse:0.53151
[13]	validation_0-rmse:0.44247	validation_1-rmse:0.51285
[14]	validation_0-rmse:0.42939	validation_1-rmse:0.49801
[15]	validation_0-rmse:0.41839	validation_1-rmse:0.48522
[16]	validation_0-rmse:0.40919	validation_1-rmse:0.47441
[17]	validation_0-rmse:0.40143	validation

### Retrain final model on full dataset

In [7]:
# Use full data for final model
df_x_full = df_x.copy()
df_y_full = df_y.copy()

final_model = XGBRegressor(
    n_estimators=best_n_estimators,
    max_depth=3,
    learning_rate=0.1,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42
)

final_model.fit(df_x_full, df_y_full.values.ravel())


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=48,
             n_jobs=None, num_parallel_tree=None, ...)

### Make predictions on full data and compute residuals

In [8]:
# Predict and calculate residuals
y_pred_full = final_model.predict(df_x_full)
residuals_full = y_pred_full - df_y_full["Brouwer mean motion"].values

df_result = df_y_full.copy()
df_result["predicted"] = y_pred_full
df_result["residuals"] = residuals_full


###  Plot Observed vs Predicted (Full Time Range)

In [9]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["Brouwer mean motion"],
    mode='lines',
    name='Observed'
))

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["predicted"],
    mode='lines',
    name='Predicted'
))

fig.update_layout(
    title='Observed vs Predicted Brouwer Mean Motion (Full Series)',
    xaxis_title='Time',
    yaxis_title='Brouwer Mean Motion (scaled)',
    template='plotly_white',
    width=1200
)

fig.show()


### Plot residuals over time 

In [10]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals'
))

fig.add_hline(y=0, line_dash="dot", line_color="black")

fig.update_layout(
    title='Residuals Over Time',
    xaxis_title='Time',
    yaxis_title='Residual (Observed - Predicted)',
    template='plotly_white',
    width= 1400
)

fig.show()


### Plot Residuals with Ground Truth Maneuvers

In [11]:
# Load maneuver data
ground_truth_df = pd.read_csv(
    "/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/cleaned maneuver file/cleaned_FENGYUN-2H.csv",
    parse_dates=["Start_Timestamp", "End_Timestamp"]
)

for col in ["Start_Timestamp", "End_Timestamp"]:
    if ground_truth_df[col].dt.tz is None:
        ground_truth_df[col] = ground_truth_df[col].dt.tz_localize("UTC")
    else:
        ground_truth_df[col] = ground_truth_df[col].dt.tz_convert("UTC")
        
# Filter to full range
test_start = df_result.index.min()
test_end = df_result.index.max()

ground_truth_df_test = ground_truth_df[
    (ground_truth_df["Start_Timestamp"] >= test_start) &
    (ground_truth_df["Start_Timestamp"] <= test_end)
]

# Get residual range
y_min = df_result["residuals"].min()
y_max = df_result["residuals"].max()

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}'
))

fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left"
)

# Ground truth maneuver lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    hover_text = f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[y_min, y_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[hover_text, hover_text],
        showlegend=False
    ))

fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))

fig.update_layout(
    title='Residuals with Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Residual (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=600,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()


#### Detect Anomalies (Using 3σ Rule)

You're detecting anomalies by looking at how far the prediction errors (residuals) deviate from what's considered normal. First, you calculate the average residual and how much they typically vary (standard deviation). Then, you set a threshold range — anything beyond 3 standard deviations above or below the mean is flagged as an anomaly. This is because such extreme values are very rare in normal behavior. So, if the model makes a prediction that’s way off from what’s expected, it’s marked as an anomaly and the timestamp is recorded.

In [ ]:
import numpy as np

# Calculate mean and standard deviation of residuals
residual_mean = df_result["residuals"].mean()
residual_std = df_result["residuals"].std()
 
# Define upper and lower thresholds
upper = residual_mean + 3 * residual_std
lower = residual_mean - 3 * residual_std

# Flag anomalies: residuals that fall outside the threshold range
df_result["anomaly"] = (df_result["residuals"] > upper) | (df_result["residuals"] < lower)
print(f"Anomaly threshold range: {lower:.4f} to {upper:.4f}")
print(" Anomaly timestamps:")
print(df_result[df_result["anomaly"]].index)


Anomaly threshold range: -0.9112 to 0.9097
 Anomaly timestamps:
DatetimeIndex(['2019-01-22 02:10:43.620959+00:00',
               '2019-04-21 14:39:28.064735+00:00',
               '2019-07-12 21:05:32.589888+00:00',
               '2019-10-01 03:15:57.591935+00:00',
               '2020-01-03 11:05:33.814751+00:00',
               '2020-04-20 14:16:42.795264+00:00',
               '2020-08-01 01:49:55.382016+00:00',
               '2020-08-02 03:41:42.695231+00:00',
               '2020-08-06 19:39:58.518720+00:00',
               '2020-08-11 01:17:58.007040+00:00',
               '2020-11-11 01:01:42.704832+00:00',
               '2021-03-09 09:09:42.494112+00:00',
               '2021-06-12 02:52:43.843296+00:00',
               '2021-09-28 03:25:47.565696+00:00',
               '2022-01-18 09:28:39.893951+00:00',
               '2022-01-19 11:18:53.603999+00:00'],
              dtype='datetime64[ns, UTC]', freq=None)


#### Plotting detected anomalies vs ground truth

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd


ground_truth_df = pd.read_csv(
    "/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/cleaned maneuver file/cleaned_FENGYUN-2H.csv",
    parse_dates=["Start_Timestamp", "End_Timestamp"]
)
test_start = df_result.index.min()
test_end = df_result.index.max()

ground_truth_df_test = ground_truth_df[
    (ground_truth_df["Start_Timestamp"] >= test_start) &
    (ground_truth_df["Start_Timestamp"] <= test_end)
]
residuals_min = df_result["residuals"].min()
residuals_max = df_result["residuals"].max()


fig = make_subplots(specs=[[{"secondary_y": True}]])

# Plot observed values 
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["Brouwer mean motion"],
    mode='lines+markers',
    name='Observed',
    marker=dict(size=4),
), secondary_y=False)

# Plot predicted values 
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["predicted"],
    mode='lines+markers',
    name='Predicted',
    marker=dict(size=4),
), secondary_y=False)

# Plot residuals 
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
), secondary_y=True)

# Plot detected anomalies
fig.add_trace(go.Scatter(
    x=df_result[df_result["anomaly"]].index,
    y=df_result[df_result["anomaly"]]["residuals"],
    mode='markers',
    name='Detected Anomalies',
    marker=dict(color='red', size=10, symbol='circle'),
    hovertemplate='Anomaly Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}',
), secondary_y=True)

# Plot ground truth maneuver lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[residuals_min, residuals_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"] * 2,
        showlegend=False
    ))
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left",
    secondary_y=True
)
fig.update_layout(
    title='Observed vs Predicted with Residuals, Detected Anomalies, and Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Brouwer Mean Motion (scaled)',
    yaxis2_title='Residuals (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=700,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    xaxis_range=[test_start, test_end]
)

fig.show()


### Residuals with detected anomaly and ground truth maneuver

In [ ]:


# Residual range for drawing maneuver lines
residuals_min = df_result["residuals"].min()
residuals_max = df_result["residuals"].max()

# Create figure
fig = go.Figure()

# Residuals
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
    line=dict(color='blue')
))

#  Detected Anomalies
fig.add_trace(go.Scatter(
    x=df_result[df_result["anomaly"]].index,
    y=df_result[df_result["anomaly"]]["residuals"],
    mode='markers',
    name='Detected Anomalies',
    marker=dict(color='red', size=10, symbol='circle'),
    hovertemplate='Anomaly Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}'
))

# Ground Truth Maneuver Lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[residuals_min, residuals_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"] * 2,
        showlegend=False
    ))
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left"
)
fig.update_layout(
    title='Residuals with Detected Anomalies and Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Residuals (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=600,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    xaxis_range=[df_result.index.min(), df_result.index.max()]
)

fig.show()
